<a href="https://colab.research.google.com/github/armandochernandez-ai/Curso-python-slava/blob/main/CUCEA/MEJORA2_FRUTAS_HORTALIZAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install selenium
import pandas as pd
import time
import re
import os
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import logging
import sys

# Configuración para Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("✅ Ejecutando en Google Colab")
except:
    IN_COLAB = False
    print("❌ No se detectó Google Colab")

# Configuración de logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class SNIIMExtractorOptimizado:
    def __init__(self):
        self.base_url = "https://www.economia-sniim.gob.mx/nuevo/Home.aspx?opcion=Consultas/MercadosNacionales/PreciosDeMercado/Agricolas/ConsultaFrutasYHortalizas.aspx?SubOpcion=4%7C0"
        self.driver = None
        self.wait = None
        self.session_requests = 0
        self.max_requests_before_reset = 50

    def setup_driver(self):
        """Configura el WebDriver de Selenium optimizado"""
        try:
            if IN_COLAB:
                print("🔧 Configurando ChromeDriver optimizado para Colab...")
                !apt-get update > /dev/null 2>&1
                !apt-get install -y chromium-chromedriver > /dev/null 2>&1

                chrome_options = Options()
                chrome_options.add_argument('--headless')
                chrome_options.add_argument('--no-sandbox')
                chrome_options.add_argument('--disable-dev-shm-usage')
                chrome_options.add_argument('--disable-gpu')
                chrome_options.add_argument('--window-size=1920,1080')
                chrome_options.add_argument('--user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
                chrome_options.add_argument('--disable-blink-features=AutomationControlled')
                chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
                chrome_options.add_experimental_option('useAutomationExtension', False)
                chrome_options.page_load_strategy = 'eager'

                self.driver = webdriver.Chrome(options=chrome_options)
                self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
            else:
                chrome_options = Options()
                chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
                chrome_options.page_load_strategy = 'eager'
                self.driver = webdriver.Chrome(options=chrome_options)

            self.wait = WebDriverWait(self.driver, 20)
            print("✅ ChromeDriver configurado correctamente")
            return True

        except Exception as e:
            logger.error(f"❌ Error configurando ChromeDriver: {e}")
            return False

    def navigate_to_form(self):
        """Navega al formulario principal - OPTIMIZADO"""
        try:
            print("🔄 Navegando al formulario (optimizado)...")

            if self.driver.current_url != self.base_url:
                self.driver.get(self.base_url)
                time.sleep(5)

            iframes = self.driver.find_elements(By.TAG_NAME, "iframe")
            if not iframes:
                print("❌ No se encontraron iframes")
                return False

            self.driver.switch_to.frame(iframes[0])
            time.sleep(2)

            try:
                self.driver.find_element(By.ID, "txtFechaInicio")
                print("✅ Formulario cargado correctamente")
                return True
            except:
                print("❌ Formulario no accesible")
                return False

        except Exception as e:
            logger.error(f"❌ Error navegando al formulario: {e}")
            return False

    def get_initial_page(self):
        return self.navigate_to_form()

    def set_fechas_rapido(self, fecha_inicio, fecha_fin):
        """Establece fechas de forma más rápida"""
        try:
            script = f"""
            document.getElementById('txtFechaInicio').value = '{fecha_inicio.strftime('%d/%m/%Y')}';
            document.getElementById('txtFechaFinal').value = '{fecha_fin.strftime('%d/%m/%Y')}';
            """
            self.driver.execute_script(script)
            print(f"📅 Fechas establecidas: {fecha_inicio.strftime('%d/%m/%Y')} - {fecha_fin.strftime('%d/%m/%Y')}")
            return True
        except Exception as e:
            logger.error(f"❌ Error estableciendo fechas: {e}")
            return False

    def get_productos_rapido(self):
        """Extrae productos de forma más rápida"""
        try:
            selectors = [
                "ctl00_ContentPlaceHolder1_ddlProducto",
                "ddlProducto"
            ]

            for selector in selectors:
                try:
                    dropdown = self.driver.find_element(By.ID, selector)
                    select_obj = Select(dropdown)
                    options = select_obj.options

                    productos = []
                    for i, option in enumerate(options):
                        value = option.get_attribute("value")
                        text = option.text.strip()

                        if value and value != "" and value != "-1" and text and text != "Seleccione" and text != "Todos":
                            productos.append({
                                'id': value,
                                'nombre': text,
                                'index': i
                            })

                    print(f"✅ {len(productos)} productos encontrados")
                    return productos
                except:
                    continue

            return []

        except Exception as e:
            logger.error(f"❌ Error extrayendo productos: {e}")
            return []

    def configurar_consulta_rapida(self, producto_id, tipo_precio_id):
        """Configura producto y tipo de precio de forma optimizada"""
        try:
            product_selectors = ["ctl00_ContentPlaceHolder1_ddlProducto", "ddlProducto"]
            for selector in product_selectors:
                try:
                    dropdown = self.driver.find_element(By.ID, selector)
                    Select(dropdown).select_by_value(producto_id)
                    break
                except:
                    continue

            time.sleep(1)

            precio_selectors = ["ctl00_ContentPlaceHolder1_ddlTipoPrecio", "ddlTipoPrecio"]
            for selector in precio_selectors:
                try:
                    dropdown = self.driver.find_element(By.ID, selector)
                    Select(dropdown).select_by_value(tipo_precio_id)
                    break
                except:
                    continue

            time.sleep(1)

            self.set_todos_origenes_destinos_rapido()

            return True

        except Exception as e:
            logger.error(f"❌ Error configurando consulta: {e}")
            return False

    def set_todos_origenes_destinos_rapido(self):
        """Configura origen/destino de forma rápida"""
        try:
            script = """
            try {
                var origenSelect = document.getElementById('ctl00_ContentPlaceHolder1_ddlOrigen') || document.getElementById('ddlOrigen');
                if (origenSelect) {
                    for (var i = 0; i < origenSelect.options.length; i++) {
                        if (origenSelect.options[i].value === '-1' || origenSelect.options[i].text === 'Todos') {
                            origenSelect.selectedIndex = i;
                            break;
                        }
                    }
                }
            } catch(e) {}

            try {
                var destinoSelect = document.getElementById('ctl00_ContentPlaceHolder1_ddlDestino') || document.getElementById('ddlDestino');
                if (destinoSelect) {
                    for (var i = 0; i < destinoSelect.options.length; i++) {
                        if (destinoSelect.options[i].value === '-1' || destinoSelect.options[i].text === 'Todos') {
                            destinoSelect.selectedIndex = i;
                            break;
                        }
                    }
                }
            } catch(e) {}
            """
            self.driver.execute_script(script)
            return True
        except:
            return False

    def hacer_consulta_rapida(self):
        """Ejecuta consulta de forma optimizada"""
        try:
            boton_selectors = [
                "ctl00_ContentPlaceHolder1_btnBuscar",
                "btnBuscar",
                "//input[@type='submit' and contains(@value, 'Buscar')]"
            ]

            for selector in boton_selectors:
                try:
                    if selector.startswith("//"):
                        boton = self.driver.find_element(By.XPATH, selector)
                    else:
                        boton = self.driver.find_element(By.ID, selector)

                    boton.click()
                    break
                except:
                    continue

            time.sleep(5)

            page_source = self.driver.page_source
            if "No se encontraron registros" in page_source:
                print("ℹ️ No se encontraron registros")
                return True
            elif "gvResultados" in page_source:
                return True
            else:
                return True

        except Exception as e:
            logger.error(f"❌ Error en consulta: {e}")
            return False

    def extraer_datos_tabla_rapido(self, producto_id, tipo_precio_id, producto_nombre, tipo_precio_nombre):
        """Extrae datos de tabla de forma más rápida, incluyendo múltiples páginas"""
        try:
            datos = []

            if "No se encontraron registros" in self.driver.page_source:
                return datos

            # Función para extraer datos de una página específica
            def extraer_pagina_actual():
                page_datos = []
                tabla_selectors = [
                    "ctl00_ContentPlaceHolder1_gvResultados",
                    "gvResultados",
                    "//table[contains(@id, 'Resultados')]"
                ]

                tabla_element = None
                for selector in tabla_selectors:
                    try:
                        if selector.startswith("//"):
                            tabla_element = self.driver.find_element(By.XPATH, selector)
                        else:
                            tabla_element = self.driver.find_element(By.ID, selector)
                        break
                    except:
                        continue

                if not tabla_element:
                    return page_datos

                filas = tabla_element.find_elements(By.TAG_NAME, "tr")

                start_index = 0
                if len(filas) > 0:
                    primera_fila = filas[0]
                    th_cells = primera_fila.find_elements(By.TAG_NAME, "th")
                    if th_cells:
                        start_index = 1

                for i in range(start_index, len(filas)):
                    try:
                        fila = filas[i]
                        celdas = fila.find_elements(By.TAG_NAME, "td")

                        if len(celdas) >= 6:
                            fecha = celdas[0].text.strip()

                            if not re.match(r'\d{1,2}/\d{1,2}/\d{4}', fecha):
                                continue

                            dato = {
                                'producto_id': producto_id,
                                'producto_nombre': producto_nombre,
                                'tipo_precio_id': tipo_precio_id,
                                'tipo_precio_nombre': tipo_precio_nombre,
                                'fecha': fecha,
                                'presentacion': celdas[1].text.strip() if len(celdas) > 1 else '',
                                'origen': celdas[2].text.strip() if len(celdas) > 2 else '',
                                'destino': celdas[3].text.strip() if len(celdas) > 3 else '',
                                'precio_min': self.limpiar_precio(celdas[4].text.strip() if len(celdas) > 4 else ''),
                                'precio_max': self.limpiar_precio(celdas[5].text.strip() if len(celdas) > 5 else ''),
                                'precio_frec': self.limpiar_precio(celdas[6].text.strip() if len(celdas) > 6 else ''),
                                'observaciones': celdas[7].text.strip() if len(celdas) > 7 else '',
                                'fecha_consulta': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                            }

                            if dato['fecha'] and dato['presentacion']:
                                page_datos.append(dato)

                    except Exception as e:
                        continue

                return page_datos

            # Extraer primera página
            datos_pagina = extraer_pagina_actual()
            datos.extend(datos_pagina)
            print(f"   📄 Página 1: {len(datos_pagina)} registros")

            # Detectar y navegar por páginas adicionales
            pagina_actual = 1
            while True:
                try:
                    # Buscar enlaces de paginación
                    paginacion_selectors = [
                        "//a[contains(@href, 'Page$')]",
                        "//a[contains(@id, 'gvResultados_ctl') and contains(@id, 'lnkPage')]",
                        "//a[contains(text(), '...')]"
                    ]

                    siguiente_pagina = None
                    for selector in paginacion_selectors:
                        try:
                            enlaces = self.driver.find_elements(By.XPATH, selector)
                            for enlace in enlaces:
                                texto = enlace.text.strip()
                                if texto.isdigit() and int(texto) == pagina_actual + 1:
                                    siguiente_pagina = enlace
                                    break
                                elif texto == "..." or "Siguiente" in texto:
                                    siguiente_pagina = enlace
                                    break
                            if siguiente_pagina:
                                break
                        except:
                            continue

                    if not siguiente_pagina:
                        # Buscar por el patrón de GridView
                        try:
                            enlaces_pagina = self.driver.find_elements(By.XPATH, "//a[contains(@href, 'Page$')]")
                            for enlace in enlaces_pagina:
                                if enlace.is_displayed() and enlace.is_enabled():
                                    siguiente_pagina = enlace
                                    break
                        except:
                            pass

                    if siguiente_pagina and siguiente_pagina.is_displayed():
                        pagina_actual += 1
                        print(f"   🔄 Navegando a página {pagina_actual}...")

                        # Hacer clic en la siguiente página
                        self.driver.execute_script("arguments[0].click();", siguiente_pagina)
                        time.sleep(3)

                        # Extraer datos de la nueva página
                        datos_pagina = extraer_pagina_actual()
                        if datos_pagina:
                            datos.extend(datos_pagina)
                            print(f"   📄 Página {pagina_actual}: {len(datos_pagina)} registros")
                        else:
                            print(f"   ⚠️ Página {pagina_actual} sin datos, terminando paginación")
                            break
                    else:
                        print(f"   ✅ No hay más páginas. Total páginas procesadas: {pagina_actual}")
                        break

                except Exception as e:
                    print(f"   ❌ Error en paginación: {e}")
                    break

            return datos

        except Exception as e:
            logger.error(f"❌ Error extrayendo tabla: {e}")
            return []

    def limpiar_precio(self, precio_str):
        """Limpia y convierte el precio a float"""
        if not precio_str or precio_str.strip() == '':
            return None
        precio_limpio = re.sub(r'[^\d.]', '', precio_str)
        try:
            return float(precio_limpio) if precio_limpio else None
        except ValueError:
            return None

    def necesita_reset(self):
        """Determina si necesita resetear la sesión"""
        self.session_requests += 1
        return self.session_requests >= self.max_requests_before_reset

    def reset_suave(self):
        """Reset suave sin recargar página completa"""
        try:
            self.driver.switch_to.default_content()
            time.sleep(2)
            return self.navigate_to_form()
        except:
            return False

    def guardar_datos(self, datos, directorio_salida, archivo_nombre):
        """Guarda los datos en archivos CSV"""
        if not datos:
            return None

        os.makedirs(directorio_salida, exist_ok=True)
        df = pd.DataFrame(datos)
        archivo_csv = os.path.join(directorio_salida, f"{archivo_nombre}.csv")
        df.to_csv(archivo_csv, index=False, encoding='utf-8-sig')
        logger.info(f"💾 Datos guardados en: {archivo_csv}")
        return archivo_csv

    def close(self):
        if self.driver:
            try:
                self.driver.switch_to.default_content()
            except:
                pass
            self.driver.quit()

def configurar_directorio_colab():
    """Configura Google Drive para Colab"""
    if IN_COLAB:
        print("📁 Montando Google Drive...")
        drive.mount('/content/drive')
        directorio_base = "/content/drive/MyDrive/FRUTAS_HORTALIZAS"
        os.makedirs(directorio_base, exist_ok=True)
        print(f"✅ Directorio configurado: {directorio_base}")
        return directorio_base
    else:
        directorio_local = "FRUTAS_HORTALIZAS"
        os.makedirs(directorio_local, exist_ok=True)
        print(f"📂 Directorio local: {directorio_local}")
        return directorio_local

def main_optimizada():
    """Función principal OPTIMIZADA"""
    print("=" * 70)
    print("🌱 EXTRACTOR SNIIM - VERSIÓN OPTIMIZADA CON PAGINACIÓN")
    print("=" * 70)

    directorio_salida = configurar_directorio_colab()
    extractor = SNIIMExtractorOptimizado()

    if not extractor.setup_driver():
        print("❌ No se pudo configurar Selenium.")
        return

    try:
        print("\n🔗 Inicializando sesión...")
        if not extractor.get_initial_page():
            return

        productos = extractor.get_productos_rapido()
        if not productos:
            print("❌ No se encontraron productos.")
            return

        tipos_precio = [
            {'id': '0', 'nombre': 'Presentación Comercial (encuestado)', 'element_id': '0'},
            {'id': '1', 'nombre': 'por kilogramo (calculado)', 'element_id': '1'}
        ]

        fecha_fin = datetime.now() - timedelta(days=5)
        fecha_inicio = fecha_fin - timedelta(days=1)

        if not extractor.set_fechas_rapido(fecha_inicio, fecha_fin):
            print("❌ Error configurando fechas.")
            return

        print(f"\n📅 Rango: {fecha_inicio.strftime('%d/%m/%Y')} - {fecha_fin.strftime('%d/%m/%Y')}")
        print(f"🍎 Productos: {len(productos)}")
        print(f"💰 Tipos precio: {len(tipos_precio)}")
        print(f"🚀 INICIANDO EXTRACCIÓN CON PAGINACIÓN...")

        todos_los_datos = []
        productos_procesados = 0

        for i, producto in enumerate(productos, 1):
            print(f"\n🍎 [{i}/{len(productos)}] Procesando: {producto['nombre'][:50]}...")

            if extractor.necesita_reset():
                print("🔄 Reset suave de sesión...")
                extractor.reset_suave()
                extractor.set_fechas_rapido(fecha_inicio, fecha_fin)
                extractor.session_requests = 0

            for tipo_precio in tipos_precio:
                try:
                    if extractor.configurar_consulta_rapida(producto['id'], tipo_precio['element_id']):
                        if extractor.hacer_consulta_rapida():
                            datos = extractor.extraer_datos_tabla_rapido(
                                producto['id'], tipo_precio['id'],
                                producto['nombre'], tipo_precio['nombre']
                            )
                            todos_los_datos.extend(datos)
                            if datos:
                                print(f"   ✅ {tipo_precio['nombre'][:20]}: {len(datos)} registros totales")

                except Exception as e:
                    print(f"   ❌ Error: {e}")
                    continue

            productos_procesados += 1

            if i % 10 == 0 and todos_los_datos:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                archivo_parcial = extractor.guardar_datos(
                    todos_los_datos,
                    directorio_salida,
                    f"sniim_paginacion_parcial_{i}_{timestamp}"
                )
                print(f"💾 Parcial guardado: {archivo_parcial}")

        if todos_los_datos:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            archivo_final = extractor.guardar_datos(
                todos_los_datos,
                directorio_salida,
                f"sniim_paginacion_completo_{timestamp}"
            )

            print(f"\n{'='*80}")
            print("🎉 EXTRACCIÓN CON PAGINACIÓN COMPLETADA")
            print(f"{'='*80}")
            print(f"📊 TOTAL REGISTROS: {len(todos_los_datos):,}")
            print(f"🍎 PRODUCTOS PROCESADOS: {productos_procesados}/{len(productos)}")
            print(f"💾 ARCHIVO: {archivo_final}")

    except Exception as e:
        logger.error(f"❌ Error en ejecución: {e}")
        import traceback
        traceback.print_exc()

    finally:
        extractor.close()

if __name__ == "__main__":
    if IN_COLAB:
        !pip install selenium -q
    main_optimizada()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 25.3 MB/s eta 0:00:00
✅ Ejecutando en Google Colab
🌱 EXTRACTOR SNIIM - VERSIÓN OPTIMIZADA CON PAGINACIÓN
📁 Montando Google Drive...
Mounted at /content/drive
✅ Directorio configurado: /content/drive/MyDrive/FRUTAS_HORTALIZAS
🔧 Configurando ChromeDriver optimizado para Colab...
✅ ChromeDriver configurado correctamente

🔗 Inicializando sesión...
🔄 Navegando al formulario (optimizado)...
✅ Formulario cargado correctamente
✅ 222 productos encontrados
📅 Fechas establecidas: 30/10/2025 - 31/10/2025

📅 Rango: 30/10/2025 - 31/10/2025
🍎 Productos: 222
💰 Tipos precio: 2
🚀 INICIANDO EXTRACCIÓN CON PAGINACIÓN...

🍎 [1/222] Procesando: Acelga - Primera...
   📄 Página 1: 31 registros
   ✅ No hay más páginas. Total páginas procesadas: 1
   ✅ Presentación Comerci: 31 registros totales
   📄 Página 1: 31 registros
   ✅ No hay más páginas. Total páginas procesadas: 1
   